## Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Install/update gdown
!pip -q install -U gdown

# Shared folder link
shared_folder_url = (
    "https://drive.google.com/drive/folders/"
    "1yGmTxJaYIWUcf-ZRSW4NCh3pMOlNsT7i?usp=sharing"
)

# Destination in your own Google Drive
destination = "/content/drive/MyDrive/Colab Notebooks/talkinghead"


# Copy the entire shared folder to your Google Drive
!gdown --folder "{shared_folder_url}" -O "{destination}"

print("Copy completed.")
print("Saved to:", destination)

In [3]:
destination

'/content/drive/MyDrive/Colab Notebooks/talkinghead'

In [4]:
%cd "$destination"

/content/drive/MyDrive/Colab Notebooks/talkinghead


In [5]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

Tesla T4, 15360 MiB, 14913 MiB


In [6]:
!ls checkpoints
!ls checkpoints/ditto_pytorch
!ls checkpoints/ditto_cfg

ditto_cfg  ditto_onnx  ditto_pytorch  ditto_trt_Ampere_Plus  LICENSE  README.md
aux_models  models
v0.4_hubert_cfg_pytorch.pkl	v0.4_hubert_cfg_trt.pkl
v0.4_hubert_cfg_trt_online.pkl


In [7]:
# Install Ditto dependencies for the current Google Colab CUDA 12.8 runtime.
# Do not install the newest onnxruntime-gpu without pinning it: versions
# 1.27+ require CUDA 13, while this Colab runtime uses CUDA 12.8.

!pip uninstall -y -q onnxruntime onnxruntime-gpu

!pip install -q --no-cache-dir \
    librosa \
    tqdm \
    filetype \
    imageio \
    opencv-python-headless \
    scikit-image \
    cython \
    cuda-python \
    imageio-ffmpeg \
    colored \
    polygraphy \
    numpy==2.0.1 \
    mediapipe \
    "onnxruntime-gpu[cuda,cudnn]==1.26.0"

print("Installation finished. Restart the Colab runtime before continuing.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 326.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.0/277.0 MB 146.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 453.5/453.5 kB 385.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.5/36.5 MB 300.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 344.8 MB/s eta 0:00:00
Installation finished. Restart the Colab runtime before continuing.


### Restart the runtime

After running the installation cell, choose **Runtime → Restart session**. Then rerun the Drive mount, `%cd`, GPU check, and the cells below. A restart is necessary because NumPy, MediaPipe, and ONNX Runtime contain compiled libraries.


In [8]:
# Verify that PyTorch and ONNX Runtime use compatible CUDA 12.x libraries.
# inference.py imports torch before loading the ONNX-based helper models.
import torch
import onnxruntime as ort

print("PyTorch:", torch.__version__)
print("PyTorch CUDA:", torch.version.cuda)
print("ONNX Runtime:", ort.__version__)
print("Available providers:", ort.get_available_providers())

assert torch.cuda.is_available(), "Colab GPU is not enabled. Select a GPU runtime."
assert ort.__version__.startswith("1.26."), "Expected ONNX Runtime GPU 1.26.x for CUDA 12.8."
assert "CUDAExecutionProvider" in ort.get_available_providers(), (
    "CUDAExecutionProvider is unavailable. Restart the runtime and rerun the notebook."
)


PyTorch: 2.11.0+cu128
PyTorch CUDA: 12.8
ONNX Runtime: 1.26.0
Available providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


In [9]:
# Confirm the compiled package versions after restarting the runtime.
import numpy as np
import mediapipe as mp

print("NumPy:", np.__version__)
print("MediaPipe:", mp.__version__)


NumPy: 2.0.2
MediaPipe: 1.0.0


In [10]:
import numpy as np
import torch
import onnxruntime as ort

print("NumPy:", np.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("ONNX Runtime:", ort.__version__)
print("ONNX providers:", ort.get_available_providers())


NumPy: 2.0.2
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
ONNX Runtime: 1.26.0
ONNX providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


In [11]:
current_path=destination

In [12]:
!pwd

/content/drive/MyDrive/Colab Notebooks/talkinghead


### Run inference

The NumPy deprecated-API message and TensorFlow CPU feature message are informational. The MediaPipe `FaceLandmarker.__del__` message occurs during interpreter shutdown and does not invalidate an MP4 that was already created. The important check is that ONNX Runtime no longer reports a missing `libcublasLt.so.13` and uses `CUDAExecutionProvider`.


In [13]:
!python inference.py \
  --data_root "{current_path}/checkpoints/ditto_pytorch" \
  --cfg_pkl "{current_path}/checkpoints/ditto_cfg/v0.4_hubert_cfg_pytorch.pkl" \
  --audio_path "{current_path}/example/audio1.wav" \
  --source_path "{current_path}/example/image1.jpg" \
  --output_path "{current_path}/tmp/output.mp4"

In file included from /usr/local/lib/python3.12/dist-packages/numpy/_core/include/numpy/ndarraytypes.h:1909,
                 from /usr/local/lib/python3.12/dist-packages/numpy/_core/include/numpy/ndarrayobject.h:12,
                 from /usr/local/lib/python3.12/dist-packages/numpy/_core/include/numpy/arrayobject.h:5,
                 from /root/.pyxbld/temp.linux-x86_64-cpython-312/content/drive/MyDrive/Colab Notebooks/talkinghead/core/utils/blend/blend.c:1259:
/usr/local/lib/python3.12/dist-packages/numpy/_core/include/numpy/npy_1_7_deprecated_api.h:17:2: warning: #warning "Using deprecated NumPy API, disable it with " "#define NPY_NO_DEPRECATED_API NPY_1_7_API_VERSION" []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wcpp-Wcpp]8;;]
   17 | #warning "Using deprecated NumPy API, disable it with " \
      |  ^~~~~~~
W0000 00:00:1785807868.873122    3443 face_landmarker_graph.cc:180] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
INFO: Created Tens